# 04 - Build Gold Layer

Este notebook constrói a camada Gold a partir da tabela Silver.

## Objetivos

- Criar uma tabela de consumo geral com dados limpos de corridas.
- Criar uma tabela agregada com a média de `total_amount` por mês.
- Criar uma tabela agregada com a média de `passenger_count` por hora no mês de maio.
- Persistir os objetos Gold como tabelas Delta no Unity Catalog.

A camada Gold é orientada ao consumo analítico e responde diretamente às perguntas propostas no case.

---

## 00. Criação da camada Gold

#### 01. Libs usadas e Import do Projeto

In [0]:
import os
import sys
import importlib

PROJECT_ROOT = "/Workspace/Users/lucas12345543@gmail.com/ifood-case"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Files in src:", os.listdir(f"{PROJECT_ROOT}/src"))

for module_name in ["src.build_gold"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

import src.build_gold as gold

print("Silver table:", gold.SILVER_TABLE_NAME)
print("Gold trips table:", gold.GOLD_TRIPS_TABLE_NAME)
print("Gold monthly avg table:", gold.GOLD_MONTHLY_AVG_TABLE_NAME)
print("Gold May hourly passengers table:", gold.GOLD_MAY_HOURLY_PASSENGERS_TABLE_NAME)

from src.build_gold import create_gold

Project root: /Workspace/Users/lucas12345543@gmail.com/ifood-case
Files in src: ['__init__.py', 'ingest_landing.py', 'build_silver.py', 'build_raw.py', 'create_gold_views.py', 'build_gold.py']
Silver table: workspace.ifood_case.silver_yellow_taxi_trips
Gold trips table: workspace.ifood_case.gold_yellow_taxi_trips
Gold monthly avg table: workspace.ifood_case.gold_avg_total_amount_by_month
Gold May hourly passengers table: workspace.ifood_case.gold_avg_passenger_count_may_by_hour


#### 02. Tables Gold

In [0]:
gold_outputs = create_gold(
    spark=spark,
    mode="overwrite",
)

gold_trips_df = gold_outputs["gold_trips_df"]
monthly_avg_df = gold_outputs["monthly_avg_df"]
may_hourly_passengers_df = gold_outputs["may_hourly_passengers_df"]

print(f"Gold trips records: {gold_trips_df.count()}")

display(gold_trips_df.limit(10))

Gold trips records: 15338137


VendorID,passenger_count,total_amount,tpep_pickup_datetime,tpep_dropoff_datetime,pickup_date,pickup_year,pickup_month,pickup_day,pickup_hour,trip_duration_minutes
2,1.0,11.1,2023-03-01T00:06:43.000Z,2023-03-01T00:16:43.000Z,2023-03-01,2023,3,1,0,10.0
2,2.0,76.49,2023-03-01T00:08:25.000Z,2023-03-01T00:39:30.000Z,2023-03-01,2023,3,1,0,31.083333333333332
1,1.0,24.7,2023-03-01T00:49:37.000Z,2023-03-01T01:01:05.000Z,2023-03-01,2023,3,1,0,11.466666666666667
2,1.0,14.64,2023-03-01T00:08:04.000Z,2023-03-01T00:11:06.000Z,2023-03-01,2023,3,1,0,3.033333333333333
1,1.0,18.0,2023-03-01T00:09:09.000Z,2023-03-01T00:17:34.000Z,2023-03-01,2023,3,1,0,8.416666666666666
1,1.0,20.5,2023-03-01T00:32:21.000Z,2023-03-01T00:42:08.000Z,2023-03-01,2023,3,1,0,9.783333333333333
1,1.0,15.7,2023-03-01T00:45:12.000Z,2023-03-01T00:52:37.000Z,2023-03-01,2023,3,1,0,7.416666666666667
1,1.0,40.4,2023-03-01T00:19:43.000Z,2023-03-01T00:39:37.000Z,2023-03-01,2023,3,1,0,19.9
2,1.0,22.2,2023-03-01T00:08:42.000Z,2023-03-01T00:18:45.000Z,2023-03-01,2023,3,1,0,10.05
2,1.0,15.3,2023-03-01T00:48:06.000Z,2023-03-01T00:57:15.000Z,2023-03-01,2023,3,1,0,9.15


___

## 01. Validar tabelas Gold

#### 01. Confirmar tabela

In [0]:
display(
    spark.sql("SHOW TABLES IN workspace.ifood_case")
)

database,tableName,isTemporary
ifood_case,gold_avg_passenger_count_may_by_hour,false
ifood_case,gold_avg_total_amount_by_month,false
ifood_case,gold_yellow_taxi_trips,false
ifood_case,raw_yellow_taxi_trips,false
ifood_case,silver_yellow_taxi_trips,false


#### 02. Validar tabela Gold de viagens

In [0]:
gold_trips_table = spark.table("workspace.ifood_case.gold_yellow_taxi_trips")

print(f"Gold trips table records: {gold_trips_table.count()}")

gold_trips_table.printSchema()

display(gold_trips_table.limit(10))

Gold trips table records: 15338137
root
 |-- VendorID: integer (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_day: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)



VendorID,passenger_count,total_amount,tpep_pickup_datetime,tpep_dropoff_datetime,pickup_date,pickup_year,pickup_month,pickup_day,pickup_hour,trip_duration_minutes
2,1.0,14.3,2023-01-01T00:32:10.000Z,2023-01-01T00:40:36.000Z,2023-01-01,2023,1,1,0,8.433333333333334
2,1.0,16.9,2023-01-01T00:55:08.000Z,2023-01-01T01:01:27.000Z,2023-01-01,2023,1,1,0,6.316666666666666
2,1.0,34.9,2023-01-01T00:25:04.000Z,2023-01-01T00:37:49.000Z,2023-01-01,2023,1,1,0,12.75
2,1.0,19.68,2023-01-01T00:10:29.000Z,2023-01-01T00:21:19.000Z,2023-01-01,2023,1,1,0,10.833333333333334
2,1.0,27.8,2023-01-01T00:50:34.000Z,2023-01-01T01:02:52.000Z,2023-01-01,2023,1,1,0,12.3
2,1.0,20.52,2023-01-01T00:09:22.000Z,2023-01-01T00:19:49.000Z,2023-01-01,2023,1,1,0,10.45
2,1.0,64.44,2023-01-01T00:27:12.000Z,2023-01-01T00:49:56.000Z,2023-01-01,2023,1,1,0,22.733333333333334
2,1.0,28.38,2023-01-01T00:21:44.000Z,2023-01-01T00:36:40.000Z,2023-01-01,2023,1,1,0,14.933333333333334
2,1.0,19.9,2023-01-01T00:39:42.000Z,2023-01-01T00:50:36.000Z,2023-01-01,2023,1,1,0,10.9
2,1.0,19.68,2023-01-01T00:53:01.000Z,2023-01-01T01:01:45.000Z,2023-01-01,2023,1,1,0,8.733333333333333


___

## 02. Após essa etapa, a consulta presente em Analysis, já pode ser realizada extraindo a visão de negócio, através do valor do dado gerado pela Pipeline ETL.